### PRIMEINSURANCE — DQ INSIGHT ENGINE
- Reads : primeins.silver.dq_issues_log
- Writes: primeins.gold.dq_explanation_report

#### PURPOSE:
- Translates technical DQ logs into compliance-ready business explanations using an LLM. Every row from dq_issues_log is sent to the model with a structured prompt and the response is written back to Gold.

### Imports

In [0]:
from openai import OpenAI
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window
from datetime import datetime
import json

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

#### LLM CLIENT — Databricks Foundation Model


In [0]:
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
WORKSPACE_URL    = spark.conf.get("spark.databricks.workspaceUrl")
 
client = OpenAI(
    api_key  = DATABRICKS_TOKEN,
    base_url = f"https://{WORKSPACE_URL}/serving-endpoints"
)
 
MODEL_ID     = "databricks-gpt-oss-20b"
TARGET_TABLE = "primeins.gold.dq_explanation_report"
 
print(f"UC1 DQ Insight Engine  |  run_id  = {RUN_ID}")
print(f"                       |  model   = {MODEL_ID}")
print(f"                       |  target  = {TARGET_TABLE}")

#### DQSourceLoader
- Loads primeins.silver.dq_issues and attaches a numeric display sequence (issue_seq) ordered by severity then table_name for compliance dashboards

In [0]:

class DQSourceLoader:
    """
    Loads the silver DQ issues table and prepares rows for LLM processing.
 
    Responsibilities:
      - Validates that the source table exists before proceeding.
      - Adds issue_seq (row_number) for deterministic display ordering.
        Errors surface before Warnings — alphabetical sort on severity
        naturally puts "Error" ahead of "Warning".
      - Materialises rows via collect() — required because LLM calls are
        HTTP requests that cannot be distributed by Spark.
    """
 
    SOURCE_TABLE = "primeins.silver.dq_issues"
 
    def __init__(self, spark_session):
        self.spark = spark_session
 
    def load(self):
        """
        Returns a Python list of collected Spark Row objects, each enriched
        with issue_seq for ordering. Raises RuntimeError if the table is absent.
        """
        try:
            raw = self.spark.table(self.SOURCE_TABLE)
            print(f"Source table : {self.SOURCE_TABLE}")
            print(f"Row count    : {raw.count()}")
            print(f"Columns      : {raw.columns}\n")
            raw.show(10, truncate=False)
        except Exception as e:
            raise RuntimeError(
                f"Source table {self.SOURCE_TABLE} not found. "
                f"Run the Silver pipeline first.\n{e}"
            )
 
        # issue_id is the natural PK (UUID) — issue_seq is a numeric alias
        # added purely for readable ordering in the DQ dashboard.
        sequenced = raw.withColumn(
            "issue_seq",
            F.row_number().over(
                Window.orderBy(
                    F.col("severity"),    # "Error" sorts before "Warning" — desired
                    F.col("table_name"),
                    F.col("issue_id")     # UUID tie-break for full determinism
                )
            )
        )
 
        rows = sequenced.collect()
        print(f"Issues to process: {len(rows)}")
        return rows
 

#### DQPromptBuilder
- Owns all prompt logic: the system persona, the five-section structure, and the per-row user prompt that is sent to the LLM.

In [0]:

class DQPromptBuilder:
    """
    Builds the system prompt and per-row user prompts for the LLM.
 
    The five output sections map directly to compliance team requirements:
      FINDING         — what was found in plain English
      BUSINESS IMPACT — why it matters to the business
      ROOT CAUSE      — what caused it
      ACTION TAKEN    — what the pipeline did to fix it
      PREVENTION      — what should be done to stop recurrence
    """
 
    SYSTEM_PROMPT = """You are a data governance advisor writing for a compliance officer, not an engineer.
 
Your job is to explain data quality issues in plain business English.
 
Rules:
- Never use technical jargon (no "schema", "null", "regex", "pipeline", "DataFrame")
- Write in clear, direct sentences a non-technical reader can act on
- Use exactly these five section headers, in this order:
 
FINDING:
BUSINESS IMPACT:
ROOT CAUSE:
ACTION TAKEN:
PREVENTION:
 
Do not combine sections. Do not add a preamble or sign-off.
Respond only with the five sections."""
 
    # Maps raw severity codes → plain-English descriptions for the LLM prompt.
    # Fallback: the raw value is used as-is for any unmapped severity.
    SEVERITY_MAP = {
        "warning (fix applied)":   "The issue was automatically corrected by the system",
        "error (missing file)":    "A required data file was missing — no records were loaded",
        "error (quarantined)":     "Affected records were moved to a quarantine area for manual review",
    }
 
    def build_user_prompt(self, row) -> str:
        """
        Builds the user-turn prompt from a single dq_issues Row.
        Severity is translated via SEVERITY_MAP before injection so the
        model receives business-friendly language, not pipeline codes.
        """
        severity_text = self.SEVERITY_MAP.get(
            str(row["severity"]).lower(),
            str(row["severity"])   # fallback — use raw value if not in map
        )
 
        return f"""A data quality issue was detected in our insurance data system.
Explain it clearly for a compliance officer using the five sections.
 
SYSTEM AFFECTED    : {row['table_name']}
COLUMN AFFECTED    : {row['column_name']}
WHAT HAPPENED      : {row['issue_description']}
WHAT THE SYSTEM DID: {severity_text}
RECORDS AFFECTED   : {row['affected_records']}"""

#### DQInsightEngine
- Calls the LLM for each DQ issue row and accumulates structured LLM reasoning

In [0]:

class DQInsightEngine:
    """
    Drives the LLM call loop: one API call per DQ issue row.
 
    Wraps prompt construction (via DQPromptBuilder) and response parsing
    into a single process() method that returns a list of result dicts
    ready for Spark DataFrame creation.
    """
 
    def __init__(self, client, model_id: str, run_id: str, target_table: str):
        self.client       = client
        self.model_id     = model_id
        self.run_id       = run_id
        self.target_table = target_table
        self.builder      = DQPromptBuilder()
 
    # ------------------------------------------------------------------
    # RESPONSE PARSER
    #
    # databricks-gpt-oss-20b returns a plain string — returned as-is.
    # If a future model returns JSON blocks (reasoning + text), the
    # list-branch handles it gracefully without requiring code changes.
    # ------------------------------------------------------------------
    # def _parse_response(self, raw_content) -> str:
    #     """
    #     Extracts plain text from the model response.
    #     Handles both plain-string and list-of-block response formats.
    #     """
    #     if isinstance(raw_content, str):
    #         return raw_content.strip()
 
    #     # Fallback: list-based block response (non-default models only)
    #     if isinstance(raw_content, list):
    #         text_blocks = [
    #             b["text"] for b in raw_content
    #             if isinstance(b, dict) and b.get("type") == "text"
    #         ]
    #         if text_blocks:
    #             return "\n".join(text_blocks).strip()
 
    #     return str(raw_content).strip()

    @staticmethod
    def _extract_text(response) -> str:
        """Extract plain text from an LLM response.

        Handles two content formats:
        - Databricks Foundation Models: list of typed blocks
        - Standard OpenAI-compatible: plain string
        Skips reasoning and tool_use blocks; returns only type='text' content.
        """
        content = response.choices[0].message.content
        if isinstance(content, list):
            return " ".join(
                block.get("text", "")
                for block in content
                if isinstance(block, dict) and block.get("type") == "text"
            ).strip()
        return content.strip()
 
    def _call_llm(self, prompt: str) -> tuple[str, str]:
        """
        Sends a single prompt to the LLM.
        Returns (explanation_text, status) — status is "success" or "error".
        Errors are captured per-row so the overall batch always completes.
        """
        try:
            response = self.client.chat.completions.create(
                model       = self.model_id,
                messages    = [
                    {"role": "system", "content": self.builder.SYSTEM_PROMPT},
                    {"role": "user",   "content": prompt},
                ],
                max_tokens  = 800,
                temperature = 0.3,
            )
            return self._extract_text(response), "success"
        except Exception as e:
            return f"LLM call failed: {str(e)}", "error"
 
    def process(self, dq_rows: list) -> list[dict]:
        """
        Iterates over all collected DQ issue rows, calls the LLM for each,
        and returns a list of result dicts matching the Gold output schema.
        """
        results = []
        total   = len(dq_rows)
        print(f"Processing {total} DQ issues...\n")
 
        for i, row in enumerate(dq_rows, 1):
            print(f"  [{i}/{total}]  {row['table_name']} | {row['issue_description']}")
 
            prompt              = self.builder.build_user_prompt(row)
            explanation, status = self._call_llm(prompt)
 
            results.append({
                # ── Original issue fields (preserved for traceability) ──────
                "issue_id":           str(row["issue_id"]),
                "issue_seq":          int(row["issue_seq"]),   # numeric display order
                "table_name":         str(row["table_name"]),
                "column_name":        str(row["column_name"]),
                "issue_description":  str(row["issue_description"]),
                "severity":           str(row["severity"]),
                "affected_records":   int(row["affected_records"]),
                # ── AI-generated fields ─────────────────────────────────────
                "ai_explanation":     explanation,
                "generation_status":  status,
                "model_name":         self.model_id,
                "generated_at":       datetime.now().isoformat(),
                # ── Pipeline audit ──────────────────────────────────────────
                "_gold_run_id":       self.run_id,
            })
 
            print(f"         → {status}")
 
        print(f"\nAll {total} issues processed.")
        return results

#### DQReportWriter
- Converts the list of result dicts to a Spark DataFrame using an explicit schema, then overwrites the Gold Delta target table.

In [0]:


class DQReportWriter:
    """
    Writes the LLM-enriched DQ results to a Gold Delta table.
 
    Uses an explicit output schema to guard against type inference errors
    on UUID strings and free-text AI explanation fields. Full overwrite
    ensures idempotency — safe to re-run at any time.
    """
 
    OUTPUT_SCHEMA = T.StructType([
        T.StructField("issue_id",          T.StringType(),  nullable=False),  # UUID — not int
        T.StructField("issue_seq",         T.IntegerType(), nullable=True),   # display order
        T.StructField("table_name",        T.StringType(),  nullable=True),
        T.StructField("column_name",       T.StringType(),  nullable=True),
        T.StructField("issue_description", T.StringType(),  nullable=True),
        T.StructField("severity",          T.StringType(),  nullable=True),
        T.StructField("affected_records",  T.IntegerType(), nullable=True),
        T.StructField("ai_explanation",    T.StringType(),  nullable=True),
        T.StructField("generation_status", T.StringType(),  nullable=True),
        T.StructField("model_name",        T.StringType(),  nullable=True),
        T.StructField("generated_at",      T.StringType(),  nullable=True),
        T.StructField("_gold_run_id",      T.StringType(),  nullable=True),
    ])
 
    def __init__(self, spark_session, target_table: str):
        self.spark        = spark_session
        self.target_table = target_table
 
    def write(self, results: list) -> int:
        """
        Creates a DataFrame from results and overwrites the Gold target table.
        Returns the final row count for confirmation logging.
        """
        output_df = self.spark.createDataFrame(results, schema=self.OUTPUT_SCHEMA)
 
        (output_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(self.target_table))
 
        final_count = self.spark.table(self.target_table).count()
        print(f"\n✅  {self.target_table} — {final_count} rows written")
        return final_count
 

#### PIPELINE ORCHESTRATION

In [0]:

# Step 1 — Load silver DQ issues
loader  = DQSourceLoader(spark)
dq_rows = loader.load()
 
# Step 2 — Run LLM over all rows
engine  = DQInsightEngine(client, MODEL_ID, RUN_ID, TARGET_TABLE)
results = engine.process(dq_rows)
 
# Step 3 — Write to Gold Delta table
writer      = DQReportWriter(spark, TARGET_TABLE)
final_count = writer.write(results)

#### Sample Output validation

In [0]:
print("=" * 70)
print(f"SAMPLE OUTPUT — {TARGET_TABLE}")
print("=" * 70)
 
sample = spark.table(TARGET_TABLE).limit(3).collect()
 
for row in sample:
    print(f"\n{'─' * 70}")
    print(f"Issue ID         : {row['issue_id']}")
    print(f"Seq              : {row['issue_seq']}")
    print(f"Table            : {row['table_name']}")
    print(f"Column           : {row['column_name']}")
    print(f"Issue            : {row['issue_description']}")
    print(f"Severity         : {row['severity']}")
    print(f"Records Hit      : {row['affected_records']}")
    print(f"Generation Status: {row['generation_status']}")
    print(f"Model            : {row['model_name']}")
    print(f"Generated At     : {row['generated_at']}")
    print(f"\n{'─' * 35} AI EXPLANATION {'─' * 20}\n")
    print(row['ai_explanation'])
 
print(f"\n{'─' * 70}")
print(f"Run ID     : {RUN_ID}")
print(f"Total rows : {final_count}")
 
# Final schema confirmation (for submission screenshot)
print("\nFinal table schema:")
spark.table(TARGET_TABLE).printSchema()